# City Energy Transition Challenge — Voting dan Dashboard BAU (v0.10)

**Dua kota, dua ronde voting, lima opsi portofolio (A–E). Opsi E selalu Do Nothing.** Skenario referensi **BAU = E → E** selalu muncul, bahkan ketika kedua kota memilih investasi lain. Horizon 15 tahun, tanpa diskonto. Semua angka adalah asumsi edukatif.

Jalankan sel dari atas ke bawah. Letakkan file `city_energy_challenge_v09_engine.py` pada folder yang sama dengan notebook.

## Indikator teknis: keandalan pasokan

Dashboard selalu membandingkan **Kota A, Kota B, dan BAU (E → E)** pada tiga indikator utama: total biaya, emisi kumulatif, dan **persentase kebutuhan listrik yang terpenuhi saat gangguan 14 hari pada tahun 15**. Angka stress test tidak dicampur ke biaya atau emisi normal.

Stress test mengurangi ketersediaan PLTU dan gas menjadi 50%, PLTS 70%, PLTB 60%, PLTA 90%, geothermal 95%, biomassa 85%. Semua adalah asumsi edukatif yang dapat diubah. Nilai ini merupakan **indikator kecukupan energi agregat**, bukan indeks keandalan jaringan listrik secara probabilistik ataupun simulasi kestabilan frekuensi/tegangan. Model tidak memiliki baterai dan masih memakai tiga blok waktu.

**Penting:** bila kebutuhan listrik normal tidak terlayani, angka biaya yang lebih rendah tidak boleh langsung ditafsirkan sebagai strategi yang lebih baik.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd()))
from city_energy_challenge_v10_engine import *
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
print('Model 15 tahun siap. BAU otomatis: E -> E.')

In [ ]:
# Konfigurasi stress test dapat diedit sebelum dashboard dijalankan.
print('Konfigurasi stress test:', STRESS_TEST)
# Contoh: STRESS_TEST['availability']['coal'] = 0.40

In [ ]:
# Setiap opsi adalah kombinasi paket teknologi, bukan teknologi tunggal.
OPTIONS = {
    0: {
        'A': {'solar': 4, 'wind': 3},
        'B': {'solar': 1, 'biomass': 1, 'hydro': 1, 'geothermal': 1},
        'C': {'solar': 2, 'wind': 2, 'biomass': 1, 'hydro': 1},
        'D': {'solar': 1, 'wind': 1, 'hydro': 1, 'geothermal': 1},
        'E': {},  # DO NOTHING
    },
    5: {
        'A': {'solar': 2},
        'B': {'solar': 1, 'wind': 1},
        'C': {'biomass': 1},
        'D': {'hydro': 1},
        'E': {},  # DO NOTHING
    },
}
OPTION_NAMES = {
    0: {'A': 'Energi Cepat', 'B': 'Energi Andal', 'C': 'Diversifikasi',
        'D': 'Infrastruktur Jangka Panjang', 'E': 'Do Nothing'},
    5: {'A': 'Ekspansi PLTS', 'B': 'PLTS + PLTB', 'C': 'CHP Biomassa',
        'D': 'PLTA Bendungan', 'E': 'Do Nothing'},
}
CITIES = {'Kota A': [], 'Kota B': []}
VOTING_LOG = []
BAU_DECISIONS = {0: 'E', 5: 'E'}
BAU_INVESTMENTS = []  # E -> E; never modified by voting

def expand_portfolio(year, letter):
    return [Investment(k, v, year) for k, v in OPTIONS[year][letter].items()]

def option_table(year, city=None):
    if year not in OPTIONS: raise ValueError('Tahun harus 0 atau 5')
    if city is not None and city not in CITIES: raise ValueError('Kota tidak dikenal')
    rows = []
    for letter, mix in OPTIONS[year].items():
        capex = sum(TECH[k].capex_billion * v for k, v in mix.items())
        row = {'Opsi': letter, 'Strategi': OPTION_NAMES[year][letter],
               'Paket': ', '.join(f'{k} x{v}' for k, v in mix.items()) or 'Tidak berinvestasi',
               'CAPEX (Rp miliar)': capex}
        if city is not None:
            try:
                validate_investments(CITIES[city] + expand_portfolio(year, letter))
                row['Layak'] = 'Ya'
            except ValueError as e:
                row['Layak'] = f'Tidak: {e}'
        rows.append(row)
    return pd.DataFrame(rows)

display(option_table(0))
display(option_table(5))

## Voting dua ronde

Fasilitator dapat memasukkan pemenang voting (`winner='A'`) atau jumlah suara per opsi (`counts={'A': ..., ...}`). Jika suara tertinggi seri, program meminta voting ulang. Pilihan E tetap dicatat meskipun tidak menambah investasi.

In [ ]:
def cast_vote(city, year, counts=None, winner=None):
    if city not in CITIES or year not in OPTIONS:
        raise ValueError('Kota atau tahun tidak valid')
    if any(x['city'] == city and x['year'] == year for x in VOTING_LOG):
        raise ValueError('Voting ronde ini sudah disimpan')
    if year == 5 and not any(x['city'] == city and x['year'] == 0 for x in VOTING_LOG):
        raise ValueError('Voting tahun 0 harus dilakukan dahulu')
    if (counts is None) == (winner is None):
        raise ValueError('Isi counts ATAU winner')
    if counts is not None:
        if set(counts) != set('ABCDE') or any(type(v) is not int or v < 0 for v in counts.values()):
            raise ValueError('counts harus A-E dengan bilangan bulat nonnegatif')
        if sum(counts.values()) == 0: raise ValueError('Belum ada suara')
        max_votes = max(counts.values())
        ties = [k for k, v in counts.items() if v == max_votes]
        if len(ties) != 1: raise ValueError(f'Suara seri {ties}: voting ulang')
        winner = ties[0]
    if not isinstance(winner, str) or winner.upper() not in OPTIONS[year]:
        raise ValueError('Pemenang harus A-E')
    winner = winner.upper()
    proposed = CITIES[city] + expand_portfolio(year, winner)
    capex = validate_investments(proposed)  # no mutation before validation
    CITIES[city] = proposed
    VOTING_LOG.append({'city': city, 'year': year, 'winner': winner,
                       'votes': None if counts is None else dict(counts),
                       'capex_cumulative_billion': capex})
    print(f'{city} tahun {year}: {winner} — {OPTION_NAMES[year][winner]}; '
          f'CAPEX Rp{capex:.0f} miliar; sisa APBD Rp{BUDGET_BILLION-capex:.0f} miliar')
    return simulate(CITIES[city])

def reset_votes():
    for city in CITIES: CITIES[city] = []
    VOTING_LOG.clear()
    print('Voting Kota A dan Kota B direset. BAU tetap E -> E.')

# BAU selalu tersedia, bahkan sebelum voting dimulai.
assert OPTIONS[0]['E'] == OPTIONS[5]['E'] == {}
assert simulate(BAU_INVESTMENTS)['capex_billion'] == 0
print('BAU permanen aktif: E -> E.')

## Dashboard: Kota A, Kota B, dan BAU

Perbandingan utama: **total biaya = CAPEX + OPEX kumulatif 15 tahun**, **emisi kumulatif 15 tahun**, dan **keandalan pasokan saat stress test**. Nilai selisih dihitung relatif terhadap BAU. Tidak ada skor gabungan atau pemenang otomatis. Perhatikan pula kebutuhan listrik yang tidak terlayani.

In [ ]:
def scenario_results(events=None, coal_settings=None):
    # Events and coal settings are identical for all scenarios.
    return {
        'Kota A': simulate(CITIES['Kota A'], events=events, coal_settings=coal_settings),
        'Kota B': simulate(CITIES['Kota B'], events=events, coal_settings=coal_settings),
        'BAU': simulate(BAU_INVESTMENTS, events=events, coal_settings=coal_settings),
    }

def comparison_table(events=None, coal_settings=None):
    scenarios = scenario_results(events, coal_settings)
    baseline = scenarios['BAU']
    bau_cost = baseline['capex_billion'] + baseline['cumulative_operating_cost_billion']
    bau_emission = baseline['cumulative_emissions_ktco2']
    rows = []
    for name, r in scenarios.items():
        last = r['annual'][-1]
        cost = r['capex_billion'] + r['cumulative_operating_cost_billion']
        unserved = sum(x['electricity_unserved_gwh'] for x in r['annual'])
        resilience = stress_test(BAU_INVESTMENTS if name == 'BAU' else CITIES[name],
                                 events=events, coal_settings=coal_settings)
        rows.append({
            'Skenario': name,
            'Keandalan stress test (%)': resilience['served_percent'],
            'ENS stress test (GWh)': resilience['unserved_gwh'],
            'CAPEX (Rp miliar)': r['capex_billion'],
            'OPEX 15 th (Rp miliar)': r['cumulative_operating_cost_billion'],
            'Total biaya 15 th (Rp miliar)': cost,
            'Penghematan vs BAU (Rp miliar)': bau_cost - cost,
            'Emisi 15 th (ktCO2)': r['cumulative_emissions_ktco2'],
            'Pengurangan emisi vs BAU (ktCO2)': bau_emission - r['cumulative_emissions_ktco2'],
            'RE listrik tahun 15 (%)': 100 * last['electricity_re_used_gwh'] / last['electricity_demand_gwh'],
            'Listrik tak terlayani 15 th (GWh)': unserved,
            'Sisa APBD (Rp miliar)': BUDGET_BILLION - r['capex_billion'],
        })
    return pd.DataFrame(rows).round(2)

def show_results(events=None, coal_settings=None):
    df = comparison_table(events, coal_settings)
    display(df)
    scenarios = scenario_results(events, coal_settings)
    names = list(scenarios)
    costs = [scenarios[n]['capex_billion'] + scenarios[n]['cumulative_operating_cost_billion'] for n in names]
    emissions = [scenarios[n]['cumulative_emissions_ktco2'] for n in names]
    reliability = [stress_test(BAU_INVESTMENTS if name == 'BAU' else CITIES[name],
                               events=events, coal_settings=coal_settings)['served_percent']
                   for name in names]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(names, reliability)
    ax.set(title='Ketahanan pasokan: gangguan 14 hari pada tahun 15',
           ylabel='Kebutuhan listrik terpenuhi (%)', ylim=(0, 105))
    ax.grid(axis='y', alpha=.2)
    plt.show()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(names, costs)
    ax.set(title='Total biaya kumulatif 15 tahun', ylabel='Rp miliar')
    ax.grid(axis='y', alpha=.2)
    plt.show()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(names, emissions)
    ax.set(title='Emisi CO2 kumulatif 15 tahun', ylabel='ktCO2')
    ax.grid(axis='y', alpha=.2)
    plt.show()
    fig, ax = plt.subplots(figsize=(8, 5))
    for name, x, y in zip(names, costs, emissions):
        ax.scatter(x, y, s=100)
        ax.annotate(name, (x, y), xytext=(6, 6), textcoords='offset points')
    ax.set(xlabel='Total biaya 15 tahun (Rp miliar)', ylabel='Emisi 15 tahun (ktCO2)',
           title='Trade-off biaya–emisi (kiri-bawah lebih baik)')
    ax.grid(alpha=.25)
    plt.show()
    fig, ax = plt.subplots(figsize=(9, 4))
    for name, r in scenarios.items():
        annual = r['annual']
        ax.plot([x['year'] for x in annual],
                [100*x['electricity_re_used_gwh']/x['electricity_demand_gwh'] for x in annual],
                marker='.', label=name)
    ax.set(xlabel='Tahun', ylabel='Pangsa listrik terbarukan (%)',
           title='Perkembangan bauran listrik selama 15 tahun')
    ax.legend(); ax.grid(alpha=.25)
    plt.show()
    if any(df['Listrik tak terlayani 15 th (GWh)'] > 0):
        print('PERHATIAN: Ada listrik tidak terlayani. Biaya rendah tidak berarti layanan setara.')
    return df

# Dashboard BAU tampil sejak awal, sekalipun belum ada voting.
show_results()

## Ronde 1 — tahun 0

Masukkan hasil voting setelah peserta memilih A–E. Contoh berikut masih dinonaktifkan agar fasilitator dapat memilih hasil sebenarnya.

In [ ]:
# cast_vote('Kota A', 0, winner='A')
# cast_vote('Kota B', 0, winner='B')
# show_results()

## Ronde 2 — tahun 5

Tampilkan kelayakan opsi berdasarkan sisa anggaran dan sumber daya masing-masing kota sebelum voting kedua.

In [ ]:
display(option_table(5, 'Kota A'))
display(option_table(5, 'Kota B'))
# cast_vote('Kota A', 5, winner='E')  # Do Nothing
# cast_vote('Kota B', 5, winner='C')
# show_results()

## Antarmuka voting opsional (ipywidgets)

Masukkan jumlah suara A–E dan simpan keputusan. Jika `ipywidgets` tidak tersedia, gunakan `cast_vote()` langsung.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    city_w = widgets.Dropdown(options=list(CITIES), description='Kota')
    year_w = widgets.Dropdown(options=[0, 5], description='Tahun')
    votes_w = {k: widgets.BoundedIntText(value=0, min=0, max=100000,
               description=k, layout=widgets.Layout(width='130px')) for k in 'ABCDE'}
    button = widgets.Button(description='Simpan voting', button_style='success')
    out = widgets.Output()
    def submit(_):
        with out:
            clear_output(wait=True)
            try:
                display(option_table(year_w.value, city_w.value))
                cast_vote(city_w.value, year_w.value,
                          counts={k: v.value for k, v in votes_w.items()})
                show_results()
            except ValueError as exc:
                print('Keputusan belum disimpan:', exc)
    button.on_click(submit)
    display(widgets.VBox([widgets.HBox([city_w, year_w]),
            widgets.HBox(list(votes_w.values())), button, out]))
except ImportError:
    print('ipywidgets tidak tersedia. Gunakan cast_vote(..., winner=...).')

### Asumsi dan keterbatasan

- E = Do Nothing pada **kedua** ronde. BAU = E → E, tetap tersedia setiap saat.
- BAU tidak menerima penambahan kapasitas fosil gratis; kekurangan pasokan ditampilkan eksplisit.
- Semua kota menggunakan asumsi permintaan, biaya, emisi, dan kejadian yang sama.
- Tiga blok waktu per hari tidak menangkap puncak jam-jaman atau musim secara rinci.
- Biaya dan emisi adalah estimasi edukatif tanpa diskonto. Sisa APBD tidak dikurangkan dari biaya sistem.
- Tidak ada penentuan pemenang otomatis; interpretasi trade-off menjadi bagian diskusi.